In [ ]:
%run Setting_Env_Variables_p2.py
google_project_id = %env GOOGLE_CLOUD_PROJECT
bucket = os.getenv("WORKSPACE_BUCKET")

import os
import subprocess
import pandas as pd
from datetime import datetime

import hail as hl
hl.init(gcs_requester_pays_configuration=google_project_id) #, log=f'{bucket}/hail_logs')
hl.default_reference(new_default_reference = "GRCh38")


In [ ]:
# !gcloud storage ls --billing-project=$GOOGLE_PROJECT gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat

In [ ]:
auxiliary_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux"
print(f'aux path: {auxiliary_path}')
vat_path = f'{auxiliary_path}/vat/*.gz'
print(f'vat path: {vat_path}')
vt_path = f'{bucket}'


mt_wgs_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"
flagged_samples = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
array_path = "gs://vwb-aou-datasets-controlled/v9/microarray/hail.mt"
# this path is in the official ct v8 fw
bed_path = "gs://aou-tutorial-notebooks-wb-sunny-radish-6214/genomic_test_data/random_sites_GRCh38.txt"


In [ ]:
# #VAT can be found here
# !gcloud storage ls -l --billing-project=$GOOGLE_PROJECT gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/vat

In [ ]:
# !gcloud storage cp gs://aou-tutorial-notebooks-wb-sunny-radish-6214/03_Getting_Started_with_Genomic_Data/01_Getting_Started_with_srWGS_Data/01.4_How_to_work_with_VAT_Hail.ipynb .

In [ ]:
vt = hl.read_table(f'{bucket}/hail_files/lmna_vat.ht')
vt


In [ ]:
%%time
vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True,) # impute=True)
vat_table = vat_table.add_index(name='id')
# mt = hl.read_matrix_table(mt_wgs_path)

In [ ]:
vt=hl.read_table()

In [ ]:
gene = 'LMNA'
ens_id = 'ENSG00000160789'

filt_vat = vat_table.filter(vat_table["gene_id"]==ens_id)

In [ ]:
# unique_values = filt_vat.aggregate(hl.agg.collect_as_set(filt_vat.consequence))

# unique_values

In [ ]:
filtered_ht = filt_vat.filter(hl.any(
    # filt_vat.consequence.contains('intron_variant'),
    # filt_vat.consequence.contains('non_coding_transcript_variant'),
    filt_vat.clinvar_classification.contains('pathogenic')
))

In [ ]:
# print(f'{gene} VAT length = {filt_vat.count()}')

In [ ]:
# filt_vat.write(f'{bucket}/hail_files/lmna_vat.ht', overwrite=True)
# 

In [ ]:
vat_table.describe()

In [ ]:
intervals = ['chr1:156.08M-156.15M']
filtered_mt = hl.filter_intervals(
    mt,
    [hl.parse_locus_interval(x, reference_genome='GRCh38') for x in intervals])


In [ ]:
filtered_mt.rows().select().show()

In [ ]:
filtered_mt.describe()

In [ ]:
filt_table = vat_table.filter(vat_table["gene_symbol"]==gene)

In [ ]:
# vat_table.clinvar_classification.show()
fvat_table = vat_table.filter(vat_table.id <= 5)  
# fvat_table.show()

In [ ]:
# chr1:156,082,572-156,140,081 
test_intervals = ['chr1:156.08M-156.15M']
filtered_vt = vat_table.filter(hl.literal(test_intervals).contains(vat_table.show()))
filtered_vt.describe()

In [ ]:
gene = 'LMNA'
lmna_vat = vat_table.filter(vat_table["gene_symbol"]==gene)

In [ ]:
filtered_mt = vat_table.filter((vat_table.contig == 'chr2') & (vat_table.position >= 50000) & (vat_table.position <= 100000))


7/15/26 filter by gene coords rather than gene symbol?